# 🚀 FORECASTING - INFERENCIA 2025

## 📚 1. CONFIGURACIÓN Y LIBRERÍAS

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import streamlit as st
import holidays
from IPython.display import display

## 📥 2. CARGA DE DATOS DE INFERENCIA

In [ ]:
# Definir la ruta del archivo de inferencia
path_inferencia = '../data/raw/inferencia/ventas_2025_inferencia.csv'

# Cargar los datos en el DataFrame inferencia_df
inferencia_df = pd.read_csv(path_inferencia, parse_dates=['fecha'])

# Visualizar las primeras filas
display(inferencia_df.head())

## ⚙️ 3. INGENIERÍA DE VARIABLES (PIPELINE)

In [ ]:
# 3.1 Variables Temporales
inferencia_df['dia_semana'] = inferencia_df['fecha'].dt.dayofweek
inferencia_df['año'] = inferencia_df['fecha'].dt.year
inferencia_df['mes'] = inferencia_df['fecha'].dt.month
inferencia_df['dia_mes'] = inferencia_df['fecha'].dt.day
inferencia_df['es_fin_semana'] = inferencia_df['dia_semana'].isin([5, 6])
inferencia_df['trimestre'] = inferencia_df['fecha'].dt.quarter
inferencia_df['semana_anio'] = inferencia_df['fecha'].dt.isocalendar().week.astype(int)

# 3.2 Festivos (España 2025)
es_holidays = holidays.Spain(years=[2025])
inferencia_df['es_festivo'] = inferencia_df['fecha'].apply(lambda x: x in es_holidays)

# 3.3 Black Friday y Cyber Monday 2025
inferencia_df['es_black_friday'] = (inferencia_df['fecha'] == '2025-11-28')
inferencia_df['es_cyber_monday'] = (inferencia_df['fecha'] == '2025-12-01')

print("Variables temporales y eventos especiales creados.")

In [ ]:
# 3.4 Lags y Media Móvil
def create_lag_features(df, target_col='unidades_vendidas', max_lag=7):
    df = df.copy()
    df = df.sort_values(['producto_id', 'fecha'])
    for lag in range(1, max_lag + 1):
        df[f'lag_{lag}'] = df.groupby(['producto_id', df['fecha'].dt.year])[target_col].shift(lag)
    return df

inferencia_df = create_lag_features(inferencia_df)

inferencia_df['rolling_mean_7'] = (
    inferencia_df.groupby(['producto_id', inferencia_df['fecha'].dt.year])['unidades_vendidas']
    .transform(lambda x: x.rolling(window=7, min_periods=7).mean())
)

print("Lags y medias móviles calculados.")

In [ ]:
# 3.5 Análisis de Precios
competidores = ['Amazon', 'Decathlon', 'Deporvillage']
inferencia_df['descuento_porcentaje'] = ((inferencia_df['precio_venta'] - inferencia_df['precio_base']) / inferencia_df['precio_base']) * 100
inferencia_df['descuento_porcentaje'] = inferencia_df['descuento_porcentaje'].round(2)
inferencia_df['precio_competencia'] = inferencia_df[competidores].mean(axis=1).round(2)
inferencia_df['ratio_precio'] = (inferencia_df['precio_venta'] / inferencia_df['precio_competencia']).round(4)
inferencia_df = inferencia_df.drop(columns=competidores)
print("Variables de precio calculadas.")

## 🏷️ 4. CODIFICACIÓN CATEGÓRICA (ALINEACIÓN CON ENTRENAMIENTO)

In [ ]:
# 4.1 Aplicar One-Hot Encoding
inferencia_df['nombre_h'] = inferencia_df['nombre']
inferencia_df['categoria_h'] = inferencia_df['categoria']
inferencia_df['subcategoria_h'] = inferencia_df['subcategoria']

columnas_ohe = ['nombre_h', 'categoria_h', 'subcategoria_h']
inferencia_df = pd.get_dummies(inferencia_df, columns=columnas_ohe)

# 4.2 Alineación de columnas con el set de entrenamiento
cols_entrenamiento = ['fecha', 'producto_id', 'nombre', 'categoria', 'subcategoria', 'precio_base', 'es_estrella', 'unidades_vendidas', 'precio_venta', 'ingresos', 'dia_semana', 'año', 'mes', 'dia_mes', 'es_fin_semana', 'es_festivo', 'es_black_friday', 'es_cyber_monday', 'trimestre', 'semana_anio', 'lag_1', 'lag_2', 'lag_3', 'lag_4', 'lag_5', 'lag_6', 'lag_7', 'rolling_mean_7', 'descuento_porcentaje', 'precio_competencia', 'ratio_precio', 'nombre_h_Adidas Own The Run Jacket', 'nombre_h_Adidas Ultraboost 23', 'nombre_h_Asics Gel Nimbus 25', 'nombre_h_Bowflex SelectTech 552', 'nombre_h_Columbia Silver Ridge', 'nombre_h_Decathlon Bandas Elásticas Set', 'nombre_h_Domyos BM900', 'nombre_h_Domyos Kit Mancuernas 20kg', 'nombre_h_Gaiam Premium Yoga Block', 'nombre_h_Liforme Yoga Pad', 'nombre_h_Lotuscrafts Yoga Bolster', 'nombre_h_Manduka PRO Yoga Mat', 'nombre_h_Merrell Moab 2 GTX', 'nombre_h_New Balance Fresh Foam X 1080v12', 'nombre_h_Nike Air Zoom Pegasus 40', 'nombre_h_Nike Dri-FIT Miler', 'nombre_h_Puma Velocity Nitro 2', 'nombre_h_Quechua MH500', 'nombre_h_Reebok Floatride Energy 5', 'nombre_h_Reebok Professional Deck', 'nombre_h_Salomon Speedcross 5 GTX', 'nombre_h_Sveltus Kettlebell 12kg', 'nombre_h_The North Face Borealis', 'nombre_h_Trek Marlin 7', 'categoria_h_Fitness', 'categoria_h_Outdoor', 'categoria_h_Running', 'categoria_h_Wellness', 'subcategoria_h_Banco Gimnasio', 'subcategoria_h_Bandas Elásticas', 'subcategoria_h_Bicicleta Montaña', 'subcategoria_h_Bloque Yoga', 'subcategoria_h_Cojín Yoga', 'subcategoria_h_Esterilla Fitness', 'subcategoria_h_Esterilla Yoga', 'subcategoria_h_Mancuernas Ajustables', 'subcategoria_h_Mochila Trekking', 'subcategoria_h_Pesa Rusa', 'subcategoria_h_Pesas Casa', 'subcategoria_h_Rodillera Yoga', 'subcategoria_h_Ropa Montaña', 'subcategoria_h_Ropa Running', 'subcategoria_h_Zapatillas Running', 'subcategoria_h_Zapatillas Trail']

for col in cols_entrenamiento:
    if col not in inferencia_df.columns:
        inferencia_df[col] = 0

inferencia_df = inferencia_df[cols_entrenamiento]
print(f"Número total de columnas tras alineación: {len(inferencia_df.columns)}")

## 💾 5. FILTRADO Y EXPORTACIÓN

In [ ]:
# 5.1 Filtrar solo Noviembre
# Se eliminan los registros de octubre y se dejan solo los de noviembre
inferencia_df_final = inferencia_df[inferencia_df['fecha'].dt.month == 11].copy()

# 5.2 Guardar
output_path = '../data/processed/inferencia_df_transformado.csv'
inferencia_df_final.to_csv(output_path, index=False)

print(f"✅ Archivo guardado en: {output_path}")
print(f"Registros de noviembre procesados: {len(inferencia_df_final)}")
display(inferencia_df_final.head())